# Chord Recognition Research

Run this notebook on **Google Colab** (Runtime → Run all, or Ctrl+F9).

What this does:
1. Installs and patches madmom (handles Python 3.10+ incompatibilities automatically)
2. Lets you upload an mp3
3. Runs madmom's CNN + CRF chord recognizer on it
4. Prints timestamped chord segments

**Ground truth comparison**: if you know the chords of the song you upload, note them down before running — comparing what you hear vs what madmom says is the main learning here.

In [1]:
# Install dependencies
# madmom needs numpy + cython available at build time, hence the order
import sys

!{sys.executable} -m pip install -q 'numpy>=1.26' cython
!{sys.executable} -m pip install -q madmom==0.16.1 --no-build-isolation
!{sys.executable} -m pip install -q librosa mir_eval soundfile

print('Install complete')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 60.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 6.9 MB/s eta 0:00:00
Install complete


In [2]:
# Patch madmom 0.16.1 for Python 3.10+ and numpy 1.24+
# madmom uses collections.MutableSequence (removed in Py3.10) and
# np.float / np.int / np.bool / np.object (removed in numpy 1.24)
import re
import sysconfig
from pathlib import Path

NUMPY_ALIASES = [
    (re.compile(r'\bnp\.float\b(?!\d)'), 'float'),
    (re.compile(r'\bnp\.int\b(?!\d|p\b)'), 'int'),
    (re.compile(r'\bnp\.bool\b(?!_)'), 'bool'),
    (re.compile(r'\bnp\.object\b(?!_)'), 'object'),
]
COLLECTIONS_FIX = (
    re.compile(r'^from collections import MutableSequence\b', re.MULTILINE),
    'from collections.abc import MutableSequence',
)

madmom_root = Path(sysconfig.get_paths()['purelib']) / 'madmom'
changed = []
for py_file in madmom_root.rglob('*.py'):
    text = py_file.read_text()
    patched = text
    for pattern, replacement in NUMPY_ALIASES + [COLLECTIONS_FIX]:
        patched = pattern.sub(replacement, patched)
    if patched != text:
        py_file.write_text(patched)
        changed.append(py_file.name)

print(f'Patched {len(changed)} file(s): {changed}')

Patched 20 file(s): ['processors.py', 'midi.py', '__init__.py', '__init__.py', 'chords.py', 'onsets.py', 'beats.py', 'tempo.py', 'notes.py', 'signal.py', 'stft.py', 'filters.py', 'midi.py', '__init__.py', 'beats_hmm.py', 'downbeats.py', 'onsets.py', 'beats.py', 'tempo.py', 'notes.py']


In [3]:
# Upload your mp3 (or wav)
from google.colab import files

uploaded = files.upload()  # opens file picker
audio_path = next(iter(uploaded.keys()))
print(f'Using: {audio_path}')

Saving Please Please Me (Remastered) CD 1 TRACK 1 (320).mp3 to Please Please Me (Remastered) CD 1 TRACK 1 (320).mp3
Using: Please Please Me (Remastered) CD 1 TRACK 1 (320).mp3


In [4]:
# Run chord recognition
from madmom.features.chords import CNNChordFeatureProcessor, CRFChordRecognitionProcessor

print(f'Analyzing {audio_path} ...')
feat = CNNChordFeatureProcessor()(audio_path)
segments = CRFChordRecognitionProcessor()(feat)

results = [(float(s), float(e), str(label)) for s, e, label in segments]
print(f'Done. {len(results)} segments detected.\n')

print(f'{"Start":>8}  {"End":>8}  Label')
print('-' * 30)
for start, end, label in results:
    duration = end - start
    print(f'{start:8.2f}  {end:8.2f}  {label}  ({duration:.2f}s)')

Analyzing Please Please Me (Remastered) CD 1 TRACK 1 (320).mp3 ...
Done. 64 segments detected.

   Start       End  Label
------------------------------
    0.00      2.60  N  (2.60s)
    2.60     11.70  E:maj  (9.10s)
   11.70     13.00  A:maj  (1.30s)
   13.00     17.60  E:maj  (4.60s)
   17.60     20.40  B:maj  (2.80s)
   20.40     23.40  E:maj  (3.00s)
   23.40     24.90  A:maj  (1.50s)
   24.90     26.40  A:min  (1.50s)
   26.40     27.90  E:maj  (1.50s)
   27.90     29.40  B:maj  (1.50s)
   29.40     35.30  E:maj  (5.90s)
   35.30     36.90  A:maj  (1.60s)
   36.90     41.50  E:maj  (4.60s)
   41.50     44.30  B:maj  (2.80s)
   44.30     47.20  E:maj  (2.90s)
   47.20     48.80  A:maj  (1.60s)
   48.80     50.30  C:maj  (1.50s)
   50.30     51.70  E:maj  (1.40s)
   51.70     53.20  B:maj  (1.50s)
   53.20     56.10  E:maj  (2.90s)
   56.10     65.40  A:maj  (9.30s)
   65.40     66.10  B:min  (0.70s)
   66.10     68.30  B:maj  (2.20s)
   68.30     71.20  A:maj  (2.90s)
   71.20   

In [5]:
# Summary statistics
from collections import Counter

total_duration = results[-1][1] if results else 0
label_durations = Counter()
for start, end, label in results:
    label_durations[label] += end - start

print(f'Total duration: {total_duration:.1f}s')
print(f'Unique labels: {len(label_durations)}')
print()
print('Label coverage (sorted by duration):')
for label, dur in sorted(label_durations.items(), key=lambda x: -x[1]):
    pct = 100 * dur / total_duration if total_duration else 0
    print(f'  {label:<12}  {dur:5.1f}s  ({pct:.1f}%)')

Total duration: 174.0s
Unique labels: 8

Label coverage (sorted by duration):
  E:maj          88.2s  (50.7%)
  A:maj          41.5s  (23.9%)
  B:maj          29.7s  (17.1%)
  N               4.9s  (2.8%)
  A:min           4.1s  (2.4%)
  C:maj           2.8s  (1.6%)
  B:min           1.7s  (1.0%)
  E:min           1.1s  (0.6%)


In [6]:
# Export results as a .chords.txt file (mir_eval-compatible, tab-separated)
# Download it to compare against ground truth later
from pathlib import Path
from google.colab import files as colab_files

stem = Path(audio_path).stem
out_name = f'{stem}.chords.txt'
with open(out_name, 'w') as f:
    for start, end, label in results:
        f.write(f'{start:.3f}\t{end:.3f}\t{label}\n')

colab_files.download(out_name)
print(f'Downloaded {out_name}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded Please Please Me (Remastered) CD 1 TRACK 1 (320).chords.txt


## What to look for

- **`N`** (no chord) — model has low confidence. Heavy `N` coverage is a signal that the model struggles with this song's timbre or density.
- **Root accuracy** — does the letter name match, even if quality (maj/min) is wrong?
- **Boundary accuracy** — do chord changes happen around the right times?
- **Failure patterns** — does it fail on specific sections? (intro, chorus, bridge?). Does it fail on specific chord types?

## Chord label format

madmom uses a simplified MIREX vocabulary:
- `C:maj`, `C:min` — C major/minor
- `C:7`, `C:maj7`, `C:min7` — seventh chords
- `N` — no chord / silence

## Next steps

After running 3-5 songs, note patterns in `FINDINGS.md` in the repo:
- What did it get right consistently?
- Where does it systematically fail?
- Does genre/tempo/production style seem to matter?

That shapes the next experiment: BTC model comparison, source separation preprocessing, or formal benchmark evaluation.